In [1]:
import os
os.environ["HUGGINGFACEHUB_API_TOKEN"] = "hibijibi"

In [2]:
!pip install -q langchain-openai langchain-core requests langchain-huggingface

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.8/84.8 kB 3.4 MB/s eta 0:00:00


In [3]:
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
import requests

In [4]:
# tool create

@tool
def multiply(a: int, b: int) -> int:
  """Given 2 numbers a and b this tool returns their product"""
  return a * b

In [5]:
print(multiply.invoke({'a':3, 'b':4}))

12


In [6]:
multiply.name

'multiply'

In [7]:
multiply.description

'Given 2 numbers a and b this tool returns their product'

In [8]:
multiply.args

{'a': {'title': 'A', 'type': 'integer'},
 'b': {'title': 'B', 'type': 'integer'}}

In [9]:
# tool binding

In [15]:
from langchain_huggingface import ChatHuggingFace, HuggingFaceEndpoint

llm = HuggingFaceEndpoint(
    repo_id="deepseek-ai/DeepSeek-R1-0528",
    task="text-generation"
)

llm = ChatHuggingFace(llm=llm)

In [16]:
llm.invoke('hi')

AIMessage(content='<think>\nHmm, the user just said "hi". Such a simple greeting to start with. \n\nOkay, first things first - this could mean anything. Maybe they\'re testing if I\'m alive, or just being polite before asking a real question. The tone feels neutral but friendly. \n\nI notice they didn\'t use any exclamation marks or emojis - probably not super excited, just casual. Should match their energy but stay open for whatever comes next. \n\nSince it\'s just "hi", best to keep it warm but brief. No need to overwhelm them with options yet. The smiley feels right here - shows I\'m approachable but not overbearing. \n\nImportant to leave space for them to lead after this. If they don\'t follow up, I can check back later. Short and sweet does it.\n</think>\n\nHello! 👋 How can I help you today? 😊', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 179, 'prompt_tokens': 4, 'total_tokens': 183}, 'model_name': 'deepseek-ai/DeepSeek-R1-0528', 'system_fingerpr

In [17]:
llm_with_tools = llm.bind_tools([multiply])

In [18]:
llm_with_tools.invoke('Hi how are you')

AIMessage(content='<think>\nWe are given a function called multiply that takes two integers a and b and returns their product.\n The user\'s message is "Hi how are you", which is a greeting and does not require the use of the multiply function.\n Since the user is just greeting, we should respond appropriately without using the function.\n</think>\n\nI\'m doing great! Thanks for asking. How about you? What can I help you with today? 😊', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 86, 'prompt_tokens': 130, 'total_tokens': 216}, 'model_name': 'deepseek-ai/DeepSeek-R1-0528', 'system_fingerprint': '', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019bc0c2-5078-73b3-9e2c-f32d11e4cb19-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 130, 'output_tokens': 86, 'total_tokens': 216})

In [19]:
query = HumanMessage('can you multiply 3 with 1000')

In [20]:
messages = [query]

In [21]:
messages

[HumanMessage(content='can you multiply 3 with 1000', additional_kwargs={}, response_metadata={})]

In [22]:
result = llm_with_tools.invoke(messages)

In [23]:
messages.append(result)

In [24]:
messages

[HumanMessage(content='can you multiply 3 with 1000', additional_kwargs={}, response_metadata={}),
 AIMessage(content='<think>\nWe are going to use the multiply function to compute 3 * 1000.\n The function requires two integers: a and b.\n Here, a is 3 and b is 1000.\n</think>\n\nI can help with that! Let me calculate it for you.', additional_kwargs={'tool_calls': [{'function': {'arguments': '{"a": 3, "b": 1000}', 'name': 'multiply', 'description': None}, 'id': 'call_gflnh33lfbu34atf0wsnt0gx', 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 81, 'prompt_tokens': 135, 'total_tokens': 216}, 'model_name': 'deepseek-ai/DeepSeek-R1-0528', 'system_fingerprint': '', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019bc0c2-a58d-7120-aefc-e809208c6e86-0', tool_calls=[{'name': 'multiply', 'args': {'a': 3, 'b': 1000}, 'id': 'call_gflnh33lfbu34atf0wsnt0gx', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 135, 'output_tokens'

In [25]:
tool_result = multiply.invoke(result.tool_calls[0])

In [26]:
tool_result

ToolMessage(content='3000', name='multiply', tool_call_id='call_gflnh33lfbu34atf0wsnt0gx')

In [27]:
messages.append(tool_result)

In [28]:
messages

[HumanMessage(content='can you multiply 3 with 1000', additional_kwargs={}, response_metadata={}),
 AIMessage(content='<think>\nWe are going to use the multiply function to compute 3 * 1000.\n The function requires two integers: a and b.\n Here, a is 3 and b is 1000.\n</think>\n\nI can help with that! Let me calculate it for you.', additional_kwargs={'tool_calls': [{'function': {'arguments': '{"a": 3, "b": 1000}', 'name': 'multiply', 'description': None}, 'id': 'call_gflnh33lfbu34atf0wsnt0gx', 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 81, 'prompt_tokens': 135, 'total_tokens': 216}, 'model_name': 'deepseek-ai/DeepSeek-R1-0528', 'system_fingerprint': '', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019bc0c2-a58d-7120-aefc-e809208c6e86-0', tool_calls=[{'name': 'multiply', 'args': {'a': 3, 'b': 1000}, 'id': 'call_gflnh33lfbu34atf0wsnt0gx', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 135, 'output_tokens'

In [29]:
llm_with_tools.invoke(messages).content

'<think>\nWe are given the result of the tool call: 3000\n The user\'s original question was to multiply 3 with 1000, and we have the result.\n Now, we should present the result to the user.\n Since the user only sees the output we generate, we should output the answer in a clear and helpful manner.\n However, note that the previous assistant output was a tool call, and the system now provides the result (3000).\n We are to respond to the user with the result.\n\n But note: the user\'s last input is "3000", which is the result of the tool call. However, in the conversation flow, the user didn\'t actually type that. \n Actually, the system is simulating the tool returning 3000. So as the assistant, we should now generate a natural language response.\n\n However, looking at the context: \n - The user asked: "can you multiply 3 with 1000"\n - We called the multiply tool with a=3, b=1000\n - The tool returned 3000\n\n Therefore, we should now reply to the user with the result.\n\n Importan

In [30]:
# tool create
from langchain_core.tools import InjectedToolArg
from typing import Annotated

@tool
def get_conversion_factor(base_currency: str, target_currency: str) -> float:
  """
  This function fetches the currency conversion factor between a given base currency and a target currency
  """
  url = f'https://v6.exchangerate-api.com/v6/c754eab14ffab33112e380ca/pair/{base_currency}/{target_currency}'

  response = requests.get(url)

  return response.json()

@tool
def convert(base_currency_value: int, conversion_rate: Annotated[float, InjectedToolArg]) -> float:
  """
  given a currency conversion rate this function calculates the target currency value from a given base currency value
  """

  return base_currency_value * conversion_rate


In [31]:
convert.args

{'base_currency_value': {'title': 'Base Currency Value', 'type': 'integer'},
 'conversion_rate': {'title': 'Conversion Rate', 'type': 'number'}}

In [32]:
get_conversion_factor.invoke({'base_currency':'USD','target_currency':'INR'})

{'result': 'success',
 'documentation': 'https://www.exchangerate-api.com/docs',
 'terms_of_use': 'https://www.exchangerate-api.com/terms',
 'time_last_update_unix': 1768435201,
 'time_last_update_utc': 'Thu, 15 Jan 2026 00:00:01 +0000',
 'time_next_update_unix': 1768521601,
 'time_next_update_utc': 'Fri, 16 Jan 2026 00:00:01 +0000',
 'base_code': 'USD',
 'target_code': 'INR',
 'conversion_rate': 90.3396}

In [33]:
convert.invoke({'base_currency_value':10, 'conversion_rate':85.16})

851.5999999999999

In [34]:
llm = HuggingFaceEndpoint(
    repo_id="deepseek-ai/DeepSeek-R1-0528",
    task="text-generation"
)

llm = ChatHuggingFace(llm=llm)

In [35]:
llm_with_tools = llm.bind_tools([get_conversion_factor, convert])

In [36]:
messages = [HumanMessage('What is the conversion factor between INR and USD, and based on that can you convert 10 inr to usd')]

In [37]:
messages

[HumanMessage(content='What is the conversion factor between INR and USD, and based on that can you convert 10 inr to usd', additional_kwargs={}, response_metadata={})]

In [38]:
ai_message = llm_with_tools.invoke(messages)

In [39]:
messages.append(ai_message)

In [40]:
ai_message.tool_calls

[{'name': 'get_conversion_factor',
  'args': {'base_currency': 'INR', 'target_currency': 'USD'},
  'id': '85bf7233a13e44d2ad5d5642d77fec96',
  'type': 'tool_call'},
 {'name': 'convert',
  'args': {'base_currency_value': 10},
  'id': '7e0a30d1f18840ed960bfa6580792c7b',
  'type': 'tool_call'}]

In [41]:
import json

for tool_call in ai_message.tool_calls:
  # execute the 1st tool and get the value of conversion rate
  if tool_call['name'] == 'get_conversion_factor':
    tool_message1 = get_conversion_factor.invoke(tool_call)
    # fetch this conversion rate
    conversion_rate = json.loads(tool_message1.content)['conversion_rate']
    # append this tool message to messages list
    messages.append(tool_message1)
  # execute the 2nd tool using the conversion rate from tool 1
  if tool_call['name'] == 'convert':
    # fetch the current arg
    tool_call['args']['conversion_rate'] = conversion_rate
    tool_message2 = convert.invoke(tool_call)
    messages.append(tool_message2)



In [42]:
messages

[HumanMessage(content='What is the conversion factor between INR and USD, and based on that can you convert 10 inr to usd', additional_kwargs={}, response_metadata={}),
 AIMessage(content='<think>\nWe are given two tools: \n  1. get_conversion_factor(base_currency, target_currency) -> returns the conversion factor (rate)\n  2. convert(base_currency_value) -> uses the conversion factor to convert the base value to target currency\n\nHowever, note that the `convert` function does not take the conversion factor as an argument. This suggests that the conversion factor must be obtained first and then used in the conversion.\n\nBut the problem: The `convert` function as described only takes `base_currency_value` and does not take the conversion factor. This might be an issue because without the factor, how can it convert?\n\nLooking at the function description for `convert`: "given a currency conversion rate this function calculates the target currency value from a given base currency value"

In [43]:
llm_with_tools.invoke(messages).content

'<think>\nWe are given two tool responses:\n1. For the get_conversion_factor tool call: \n   {\n      "result": "success", \n      ...,\n      "conversion_rate": 0.01107\n   }\n   This means the conversion factor from INR to USD is 0.01107.\n\n2. For the convert tool call: we got the result 0.11069999999999999.\n\nHowever, note that the user asked for:\n  - The conversion factor between INR and USD (which we got: 0.01107)\n  - And then to convert 10 INR to USD.\n\nWe have the conversion factor from the first tool and we used it in the second tool (we passed base_currency_value=10 and conversion_rate=0.01107). The second tool returned 0.1107 (approximately 0.1107 USD).\n\nBut note: the second tool call we made was:\n   convert(base_currency_value=10, conversion_rate=0.01107)\nand the tool returned 0.11069999999999999.\n\nWe can round this to a reasonable number of decimal places for currency (usually 2 or 4). Since USD is usually shown with 2 decimal places for dollars and cents, but so